In [1]:
import configparser
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    StructType,
    StructField,
    IntegerType,
    StringType
)

In [2]:
config = configparser.RawConfigParser()
config_path = Path.cwd() / "all_conn"

if not config.read(config_path):
    raise FileNotFoundError(f"Configuration file not found: {config_path}")

endpoint_url = config.get("conn", "endpoint_url")
aws_access_key_id = config.get("conn", "aws_access_key_id")
aws_secret_access_key = config.get("conn", "aws_secret_access_key")

In [3]:
spark = (
    SparkSession.builder
    .appName("Transform Movies Data")
    .config("spark.sql.extensions","io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog","org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", endpoint_url)
    .config("spark.hadoop.fs.s3a.access.key", aws_access_key_id)
    .config("spark.hadoop.fs.s3a.secret.key", aws_secret_access_key)
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/06 15:41:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv("s3a://tmdb-bronze/tmdb_5000_movies.csv")
)

26/07/06 15:42:03 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [6]:
df.count()

4803

In [7]:
df = (
    df
    .withColumnRenamed("id", "movie_id")
    .withColumn(
        "movie_id",
        F.col("movie_id").cast("string")
    )
)
df.printSchema()

root
 |-- budget: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- homepage: string (nullable = true)
 |-- movie_id: string (nullable = true)
 |-- keywords: string (nullable = true)
 |-- original_language: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- popularity: double (nullable = true)
 |-- production_companies: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- revenue: long (nullable = true)
 |-- runtime: double (nullable = true)
 |-- spoken_languages: string (nullable = true)
 |-- status: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- title: string (nullable = true)
 |-- vote_average: double (nullable = true)
 |-- vote_count: integer (nullable = true)



In [8]:
movies_df = df.select(
    F.col("movie_id").cast("string").alias("movie_id"),
    F.col("title").cast("string").alias("title"),
    F.col("budget").cast("double").alias("budget"),
    F.col("homepage").cast("string").alias("homepage"),
    F.col("original_language").cast("string").alias("original_language"),
    F.col("original_title").cast("string").alias("original_title"),
    F.col("overview").cast("string").alias("overview"),
    F.col("popularity").cast("float").alias("popularity"),
    F.to_date(F.col("release_date")).alias("release_date"),
    F.col("revenue").cast("double").alias("revenue"),
    F.col("runtime").cast("integer").alias("runtime"),
    F.col("status").cast("string").alias("status"),
    F.col("tagline").cast("string").alias("tagline"),
    F.col("vote_average").cast("float").alias("vote_average"),
    F.col("vote_count").cast("integer").alias("vote_count")
)

In [9]:
movies_df.printSchema()

root
 |-- movie_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- budget: double (nullable = true)
 |-- homepage: string (nullable = true)
 |-- original_language: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- popularity: float (nullable = true)
 |-- release_date: date (nullable = true)
 |-- revenue: double (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- vote_average: float (nullable = true)
 |-- vote_count: integer (nullable = true)



### genres

In [10]:
id_name_schema = ArrayType(
    StructType([    
        StructField("id", IntegerType(), True),
        StructField("name", StringType(), True)
    ])
)

In [11]:
genres_df = (
    df
    .withColumn("genres_parsed", F.from_json(F.col("genres"), id_name_schema)
    )
    .select("movie_id", F.inline_outer(F.col("genres_parsed"))
    )
)

In [12]:
genres_df.show(5, truncate=False)

+--------+---+---------------+
|movie_id|id |name           |
+--------+---+---------------+
|19995   |28 |Action         |
|19995   |12 |Adventure      |
|19995   |14 |Fantasy        |
|19995   |878|Science Fiction|
|285     |12 |Adventure      |
+--------+---+---------------+
only showing top 5 rows



In [13]:
genres_df.printSchema()

root
 |-- movie_id: string (nullable = true)
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)



## Missing Values --> genres

In [14]:
genres_df.select([
    F.count(F.when(F.col(field).isNull(), field)).alias(field)
    for field in genres_df.columns
]).show()

+--------+---+----+
|movie_id| id|name|
+--------+---+----+
|       0| 28|  28|
+--------+---+----+



In [15]:
genres_df = genres_df.fillna(
    {"id": "-9999"}
)

In [16]:
genres_df.select([
    F.count(F.when(F.col(field).isNull(), field)).alias(field)
    for field in genres_df.columns
]).show()

+--------+---+----+
|movie_id| id|name|
+--------+---+----+
|       0|  0|  28|
+--------+---+----+



In [17]:
duplicate_count = (
    genres_df.count()
    - genres_df.dropDuplicates().count()
)

print("Duplicate row numbers:", duplicate_count)

Duplicate row numbers: 0


### keywords

In [18]:
kw_df = (
    df
    .withColumn("kw_parsed", F.from_json(F.col("keywords"), id_name_schema)
    )
    .select("movie_id", F.inline_outer(F.col("kw_parsed"))
    )
)

In [19]:
kw_df.show(5)

+--------+----+-------------+
|movie_id|  id|         name|
+--------+----+-------------+
|   19995|1463|culture clash|
|   19995|2964|       future|
|   19995|3386|    space war|
|   19995|3388| space colony|
|   19995|3679|      society|
+--------+----+-------------+
only showing top 5 rows



In [20]:
kw_df.printSchema()

root
 |-- movie_id: string (nullable = true)
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)



## Missing Values --> keywords

In [21]:
kw_df.select([
    F.count(F.when(F.col(field).isNull(), field)).alias(field)
    for field in kw_df.columns
]).show()

+--------+---+----+
|movie_id| id|name|
+--------+---+----+
|       0|412| 412|
+--------+---+----+



In [22]:
kw_df = kw_df.fillna(
    {"id": "-9999"}
)

In [23]:
kw_df.select([
    F.count(F.when(F.col(field).isNull(), field)).alias(field)
    for field in kw_df.columns
]).show()

+--------+---+----+
|movie_id| id|name|
+--------+---+----+
|       0|  0| 412|
+--------+---+----+



In [24]:
duplicate_count = (
    kw_df.count()
    - kw_df.dropDuplicates().count()
)

print("Duplicate row numbers:", duplicate_count)

Duplicate row numbers: 0


### production_companies table

In [25]:
comp_df = (
    df
    .withColumn("comp_parsed", F.from_json(F.col("production_companies"), id_name_schema)
    )
    .select("movie_id", F.inline_outer(F.col("comp_parsed"))
    )
)

In [26]:
comp_df.show(5, truncate=False)

+--------+---+--------------------------------------+
|movie_id|id |name                                  |
+--------+---+--------------------------------------+
|19995   |289|Ingenious Film Partners               |
|19995   |306|Twentieth Century Fox Film Corporation|
|19995   |444|Dune Entertainment                    |
|19995   |574|Lightstorm Entertainment              |
|285     |2  |Walt Disney Pictures                  |
+--------+---+--------------------------------------+
only showing top 5 rows



In [27]:
comp_df.printSchema()

root
 |-- movie_id: string (nullable = true)
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)



In [28]:
comp_df.select([
    F.count(F.when(F.col(field).isNull(), field)).alias(field)
    for field in comp_df.columns
]).show()

+--------+---+----+
|movie_id| id|name|
+--------+---+----+
|       0|351| 351|
+--------+---+----+



In [29]:
comp_df = comp_df.fillna(
    {"id": "-9999"}
)

In [30]:
comp_df.select([
    F.count(F.when(F.col(field).isNull(), field)).alias(field)
    for field in comp_df.columns
]).show()

+--------+---+----+
|movie_id| id|name|
+--------+---+----+
|       0|  0| 351|
+--------+---+----+



In [31]:
duplicate_count = (
    comp_df.count()
    - comp_df.dropDuplicates().count()
)

print("Duplicate row numbers:", duplicate_count)

Duplicate row numbers: 0


### production_countries

In [32]:
countries_schema = ArrayType(
    StructType([    
        StructField("iso_3166_1", StringType(), True),
        StructField("name", StringType(), True)
    ])
)

In [33]:
countries_df = (
    df
    .withColumn("countries_parsed", F.from_json(F.col("production_countries"), countries_schema)
    )
    .select("movie_id", F.inline_outer(F.col("countries_parsed"))
    )
)

In [34]:
countries_df.printSchema()

root
 |-- movie_id: string (nullable = true)
 |-- iso_3166_1: string (nullable = true)
 |-- name: string (nullable = true)



In [35]:
countries_df.select([
    F.count(F.when(F.col(field).isNull(), field)).alias(field)
    for field in countries_df.columns
]).show()

+--------+----------+----+
|movie_id|iso_3166_1|name|
+--------+----------+----+
|       0|       174| 174|
+--------+----------+----+



In [36]:
countries_df = countries_df.fillna(
    {"iso_3166_1": "XX"}
)

In [37]:
countries_df.select([
    F.count(F.when(F.col(field).isNull(), field)).alias(field)
    for field in countries_df.columns
]).show()

+--------+----------+----+
|movie_id|iso_3166_1|name|
+--------+----------+----+
|       0|         0| 174|
+--------+----------+----+



In [38]:
duplicate_count = (
    countries_df.count()
    - countries_df.dropDuplicates().count()
)

print("Duplicate row numbers:", duplicate_count)

Duplicate row numbers: 0


### spoken_languages

In [39]:
lng_schema = ArrayType(
    StructType([    
        StructField("iso_639_1", StringType(), True),
        StructField("name", StringType(), True)
    ])
)

In [40]:
lng_df = (
    df
    .withColumn("lng_parsed", F.from_json(F.col("spoken_languages"), lng_schema)
    )
    .select("movie_id", F.inline_outer(F.col("lng_parsed"))
    )
)

In [41]:
lng_df.printSchema()

root
 |-- movie_id: string (nullable = true)
 |-- iso_639_1: string (nullable = true)
 |-- name: string (nullable = true)



In [42]:
lng_df.select([
    F.count(F.when(F.col(field).isNull(), field)).alias(field)
    for field in lng_df.columns
]).show()

+--------+---------+----+
|movie_id|iso_639_1|name|
+--------+---------+----+
|       0|       86|  86|
+--------+---------+----+



In [43]:
lng_df = lng_df.fillna(
    {"iso_639_1": "XX"}
)

In [44]:
lng_df.select([
    F.count(F.when(F.col(field).isNull(), field)).alias(field)
    for field in lng_df.columns
]).show()

+--------+---------+----+
|movie_id|iso_639_1|name|
+--------+---------+----+
|       0|        0|  86|
+--------+---------+----+



In [45]:
duplicate_lng = (
    lng_df.count()
    - lng_df.dropDuplicates().count()
)

print("Duplicate row numbers:", duplicate_lng)

Duplicate row numbers: 0


In [46]:
import boto3
from botocore.exceptions import ClientError

s3_client = boto3.client(
    "s3",
    endpoint_url=endpoint_url,
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key
)

bucket_name = "tmdb-silver"

try:
    s3_client.head_bucket(Bucket=bucket_name)
    print(f"{bucket_name} bucket exists.")

except ClientError as e:
    error_code = e.response.get("Error", {}).get("Code")

    if error_code in ("404", "NoSuchBucket", "NotFound"):
        s3_client.create_bucket(Bucket=bucket_name)
        print(f"{bucket_name} bucket created.")
    else:
        raise

tmdb-silver bucket exists.


## Writing to RustFS

In [48]:
spark.stop()

26/07/06 15:50:22 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-d4abcdcf-df73-449a-a917-a6a813ca42d0. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-d4abcdcf-df73-449a-a917-a6a813ca42d0
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:174)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:109)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:90)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1126)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4(ShutdownHookManager.scala:65)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4$adapted(ShutdownHookManager.scala:62)
	at scala.collection.IndexedSeqOptimized.fore